# HTM-Based Anomaly Detection for Telecom Time Series

Companion notebook for the paper:

> **A Hybrid Framework for Real-Time Data Drift and Anomaly Identification Using Hierarchical Temporal Memory and Statistical Tests**  
> Bandyopadhyay S., Bose J., Roychowdhury S.  
> IJMEMS 2025. [arXiv:2504.18599](https://arxiv.org/abs/2504.18599)

## Structure

This notebook has two parts:

**Part 1 — HTM Core (requires htm.core)**  
The full HTM implementation using Spatial Pooler + Temporal Memory.  
Installation: `pip install htm.core` (may require C++ build tools)

**Part 2 — Hybrid Statistical Detector (no external dependencies)**  
Works immediately. Combines CUSUM + Z-score + IQR + HTM-inspired anomaly likelihood.

All data is synthetic — no real telecom data is used.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

---
## Generate Synthetic Telecom Data

In [ ]:
from generate_data import generate_telecom_timeseries, generate_multi_metric_timeseries

# Single metric time series
df = generate_telecom_timeseries(n_points=2000, seed=42)
print(f"Dataset: {len(df)} rows")
print(f"Anomaly points: {df['is_anomaly'].sum()} ({df['is_anomaly'].mean()*100:.1f}%)")
df.head()

In [ ]:
# Visualise the synthetic data
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df['SignalSuccessRate'], color='steelblue', linewidth=0.8, label='Signal Success Rate')
anomaly_idx = df[df['is_anomaly'] == 1].index
axes[0].scatter(anomaly_idx, df.loc[anomaly_idx, 'SignalSuccessRate'], 
                color='red', s=8, label='Injected Anomalies', zorder=5)
axes[0].set_title('Synthetic Telecom Signal Success Rate with Injected Anomalies')
axes[0].set_ylabel('Success Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(df.index, df['is_anomaly'], alpha=0.5, color='red', label='Anomaly Region')
axes[1].set_title('Ground Truth Anomaly Labels')
axes[1].set_ylabel('Anomaly')
axes[1].set_xlabel('Time Step')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/synthetic_data.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
# PART 1: HTM Core Implementation

> **Note:** This section requires `htm.core`. Install with:
> ```
> pip install htm.core
> ```
> If installation fails, skip to Part 2 which works without any external HTM library.

### What HTM does here:
1. **Encode** each value as a Sparse Distributed Representation (SDR)
2. **Spatial Pooler** maps SDR to a fixed-size column representation
3. **Temporal Memory** learns sequences of column activations
4. **Anomaly score** = fraction of active columns that were *not* predicted
5. **Anomaly likelihood** = calibrated probability using rolling Gaussian model

In [ ]:
# Check if htm.core is available
try:
    import htm
    HTM_AVAILABLE = True
    print("htm.core is available. Running full HTM pipeline.")
except ImportError:
    HTM_AVAILABLE = False
    print("htm.core not available. Skip to Part 2 for the working implementation.")
    print("Install with: pip install htm.core")

In [ ]:
if HTM_AVAILABLE:
    import math
    from htm.bindings.sdr import SDR, Metrics
    from htm.encoders.rdse import RDSE, RDSE_Parameters
    from htm.encoders.date import DateEncoder
    from htm.bindings.algorithms import SpatialPooler, TemporalMemory
    from htm.algorithms.anomaly_likelihood import AnomalyLikelihood
    from htm.bindings.algorithms import Predictor
    print("HTM imports successful.")

In [ ]:
# HTM Parameters
# These are tuned for telecom signal success rate data
htm_parameters = {
    'enc': {
        'value': {'resolution': 0.01, 'size': 700, 'sparsity': 0.02},
        'time':  {'timeOfDay': (30, 1), 'weekend': 21}
    },
    'predictor': {'sdrc_alpha': 0.1},
    'sp': {
        'boostStrength': 3.0,
        'columnCount': 1638,
        'localAreaDensity': 0.04395604395604396,
        'potentialPct': 0.85,
        'synPermActiveInc': 0.04,
        'synPermConnected': 0.14,
        'synPermInactiveDec': 0.006
    },
    'tm': {
        'activationThreshold': 17,
        'cellsPerColumn': 13,
        'initialPerm': 0.21,
        'maxSegmentsPerCell': 128,
        'maxSynapsesPerSegment': 64,
        'minThreshold': 10,
        'newSynapseCount': 32,
        'permanenceDec': 0.1,
        'permanenceInc': 0.1
    },
    'anomaly': {
        'likelihood': {'probationaryPct': 0.1, 'reestimationPeriod': 100}
    }
}

print("HTM parameters configured.")

In [ ]:
def run_htm_pipeline(df, parameters):
    """
    Full HTM pipeline for anomaly detection.
    Requires htm.core.
    """
    if not HTM_AVAILABLE:
        raise ImportError("htm.core required. Skip to Part 2.")
    
    records = list(zip(
        pd.to_datetime(df['Timestamp']).tolist(),
        df['SignalSuccessRate'].tolist()
    ))
    
    # Encoders
    dateEncoder = DateEncoder(
        timeOfDay=parameters['enc']['time']['timeOfDay'],
        weekend=parameters['enc']['time']['weekend']
    )
    
    scalarEncoderParams = RDSE_Parameters()
    scalarEncoderParams.size = parameters['enc']['value']['size']
    scalarEncoderParams.sparsity = parameters['enc']['value']['sparsity']
    scalarEncoderParams.resolution = parameters['enc']['value']['resolution']
    scalarEncoder = RDSE(scalarEncoderParams)
    
    encodingWidth = dateEncoder.size + scalarEncoder.size
    
    # Spatial Pooler
    spParams = parameters['sp']
    sp = SpatialPooler(
        inputDimensions=(encodingWidth,),
        columnDimensions=(spParams['columnCount'],),
        potentialPct=spParams['potentialPct'],
        potentialRadius=encodingWidth,
        globalInhibition=True,
        localAreaDensity=spParams['localAreaDensity'],
        synPermInactiveDec=spParams['synPermInactiveDec'],
        synPermActiveInc=spParams['synPermActiveInc'],
        synPermConnected=spParams['synPermConnected'],
        boostStrength=spParams['boostStrength'],
        wrapAround=True
    )
    
    # Temporal Memory
    tmParams = parameters['tm']
    tm = TemporalMemory(
        columnDimensions=(spParams['columnCount'],),
        cellsPerColumn=tmParams['cellsPerColumn'],
        activationThreshold=tmParams['activationThreshold'],
        initialPermanence=tmParams['initialPerm'],
        connectedPermanence=spParams['synPermConnected'],
        minThreshold=tmParams['minThreshold'],
        maxNewSynapseCount=tmParams['newSynapseCount'],
        permanenceIncrement=tmParams['permanenceInc'],
        permanenceDecrement=tmParams['permanenceDec'],
        predictedSegmentDecrement=0.0,
        maxSegmentsPerCell=tmParams['maxSegmentsPerCell'],
        maxSynapsesPerSegment=tmParams['maxSynapsesPerSegment']
    )
    
    # Anomaly Likelihood
    anParams = parameters['anomaly']['likelihood']
    probationaryPeriod = int(math.floor(anParams['probationaryPct'] * len(records)))
    learningPeriod = int(math.floor(probationaryPeriod / 2.0))
    anomaly_history = AnomalyLikelihood(
        learningPeriod=learningPeriod,
        estimationSamples=probationaryPeriod - learningPeriod,
        reestimationPeriod=anParams['reestimationPeriod']
    )
    
    predictor_resolution = 0.001
    predictor = Predictor(steps=[1, 5], alpha=parameters['predictor']['sdrc_alpha'])
    
    # Main loop
    anomaly_scores = []
    anomaly_likelihoods = []
    
    for count, (timestamp, value) in enumerate(records):
        # Encode
        dateSDR = SDR(dateEncoder.size)
        scalarSDR = SDR(scalarEncoder.size)
        dateEncoder.encode(timestamp, dateSDR)
        scalarEncoder.encode(value, scalarSDR)
        
        encoding = SDR(encodingWidth)
        encoding.concatenate([dateSDR, scalarSDR])
        
        # Spatial Pooler
        activeColumns = SDR(sp.getColumnDimensions())
        sp.compute(encoding, True, activeColumns)
        
        # Temporal Memory
        tm.compute(activeColumns, learn=True)
        
        # Anomaly
        anomalyLikelihood = anomaly_history.anomalyProbability(value, tm.anomaly)
        anomaly_scores.append(tm.anomaly)
        anomaly_likelihoods.append(anomalyLikelihood)
        
        predictor.learn(count, tm.getActiveCells(), int(value / predictor_resolution))
    
    return np.array(anomaly_scores), np.array(anomaly_likelihoods)


if HTM_AVAILABLE:
    print("Running HTM pipeline...")
    htm_scores, htm_likelihoods = run_htm_pipeline(df, htm_parameters)
    print(f"Done. Mean anomaly score: {htm_scores.mean():.4f}")
else:
    print("Skipping HTM pipeline (htm.core not installed). Proceed to Part 2.")

---
# PART 2: Hybrid Statistical Detector

This works without `htm.core`. Uses CUSUM + Z-score + IQR + HTM-inspired anomaly likelihood.

See `statistical_drift.py` for full implementation.

In [ ]:
from statistical_drift import HybridDriftDetector, isolation_forest_baseline, evaluate

values = df['SignalSuccessRate'].values
ground_truth = df['is_anomaly'].values

# Run hybrid detector
detector = HybridDriftDetector(
    cusum_threshold=5.0,
    zscore_window=100,
    zscore_threshold=3.0,
    iqr_window=200
)

print("Running Hybrid Drift Detector...")
results_df = detector.process_series(values)
print(f"Done. Detected {results_df['is_anomaly'].sum()} anomaly points.")
results_df.head()

In [ ]:
# Evaluate
metrics = evaluate(ground_truth, results_df['anomaly_likelihood'].values)
print("Hybrid Detector Performance:")
for k, v in metrics.items():
    print(f"  {k:12s}: {v}")

In [ ]:
# Isolation Forest baseline
print("Running Isolation Forest baseline...")
if_scores = isolation_forest_baseline(values)
if_metrics = evaluate(ground_truth, if_scores)
print("\nIsolation Forest Performance:")
for k, v in if_metrics.items():
    print(f"  {k:12s}: {v}")

In [ ]:
# Comparison plot
import os
os.makedirs('figures', exist_ok=True)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Input signal
axes[0].plot(values, color='steelblue', linewidth=0.8, label='Signal Success Rate')
axes[0].fill_between(range(len(ground_truth)), 
                      ground_truth * values.min(),
                      ground_truth * values.max(),
                      alpha=0.2, color='red', label='True Anomaly')
axes[0].set_title('Input Signal')
axes[0].set_ylabel('Success Rate')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Individual detector scores
axes[1].plot(results_df['cusum_score'], color='orange', linewidth=0.8, label='CUSUM', alpha=0.8)
axes[1].plot(results_df['zscore_score'], color='green', linewidth=0.8, label='Z-score', alpha=0.8)
axes[1].plot(results_df['iqr_score'], color='purple', linewidth=0.8, label='IQR', alpha=0.8)
axes[1].set_title('Individual Detector Scores')
axes[1].set_ylabel('Score')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Anomaly likelihood
axes[2].plot(results_df['anomaly_likelihood'], color='red', linewidth=0.8, label='Hybrid Likelihood')
axes[2].plot(if_scores, color='blue', linewidth=0.8, label='Isolation Forest', alpha=0.7)
axes[2].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='Threshold (0.9)')
axes[2].set_title('Anomaly Likelihood Comparison')
axes[2].set_ylabel('Likelihood')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# Ground truth vs detected
axes[3].fill_between(range(len(ground_truth)), ground_truth, alpha=0.5, 
                      color='red', label='Ground Truth')
axes[3].fill_between(range(len(results_df)), 
                      (results_df['anomaly_likelihood'] > 0.9).astype(int),
                      alpha=0.4, color='orange', label='Hybrid Detected')
axes[3].set_title('Ground Truth vs Detected Anomalies')
axes[3].set_ylabel('Anomaly')
axes[3].set_xlabel('Time Step')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/anomaly_detection_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/anomaly_detection_results.png')

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Method': ['Hybrid (CUSUM+ZScore+IQR)', 'Isolation Forest'],
    'Precision': [metrics['precision'], if_metrics['precision']],
    'Recall': [metrics['recall'], if_metrics['recall']],
    'F1': [metrics['f1'], if_metrics['f1']],
    'Accuracy': [metrics['accuracy'], if_metrics['accuracy']],
    'Streaming': ['Yes', 'No (batch)'],
    'Requires Labels': ['No', 'No']
})

print("\nMethod Comparison:")
print(comparison.to_string(index=False))

---
## Key Advantages of the Hybrid Approach

| Property | CUSUM | Z-score | IQR | HTM (Part 1) |
|---|---|---|---|---|
| Gradual drift | Best | Poor | Poor | Good |
| Sudden spikes | Poor | Best | Good | Good |
| Non-Gaussian data | Poor | Poor | Best | Good |
| Streaming | Yes | Yes | Yes | Yes |
| No labels needed | Yes | Yes | Yes | Yes |
| Easy to install | Yes | Yes | Yes | No |

The hybrid approach combines the strengths of all three statistical methods.
HTM additionally learns temporal sequences — detecting anomalies that are unusual given recent history, not just unusual in absolute terms.

---
## Citation

```bibtex
@article{bandyopadhyay2025hybrid,
  title={A Hybrid Framework for Real-Time Data Drift and Anomaly Identification 
         Using Hierarchical Temporal Memory and Statistical Tests},
  author={Bandyopadhyay, S. and Bose, J. and Roychowdhury, S.},
  journal={International Journal of Mathematical, Engineering and Management Sciences},
  volume={10},
  number={3},
  year={2025},
  doi={10.48550/arXiv.2504.18599}
}
```